# Women Safety Edge AI Chip: Fine tune SmolLM2-135M on shoe mounted IMU actions

[![GitHub](https://img.shields.io/badge/GitHub-NandhaKishorM-blue?style=flat-square&logo=github)](https://github.com/NandhaKishorM)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-blue?style=flat-square&logo=linkedin)](https://www.linkedin.com/in/nandakishorpilicode/)

### Authors: Nandakishor M, Rajitna B, Sruthi K, Nisha Anish

This notebook fine tunes a small language model under 250M parameters on the synthetic MPU6050 dataset produced by `physics_sim.py`. The chip lives inside a shoe and triggers the safety alarm when a repeated kicking pattern is detected.

We use Q-LoRA so the same notebook runs on a free T4 in Google Colab. After training we merge the adapter, convert to gguf, then quantize to Q4_K_M so the model fits on a Pi Zero W class device (88 MB on disk, 4 to 6 tokens per second).

Stages:
1. Install dependencies
2. Upload `imu_actions_1000.jsonl`
3. Q-LoRA SFT on SmolLM2-135M-Instruct
4. Merge LoRA into the base model
5. Convert to gguf and quantize to Q4_K_M
6. End to end inference test that mimics the shoe firmware loop


## 1. Install libraries

In [ ]:
%cd /content

In [ ]:
!pip install accelerate peft bitsandbytes transformers

In [ ]:
!pip install trl==0.12.2

In [ ]:
!pip install transformers==4.57.3

In [ ]:
!pip install datasets

In [ ]:
!pip install llama-cpp-python

## 2. Upload the dataset

Upload `imu_actions_1000.jsonl` (generated from `physics_sim.py` and `generate_dataset.py`) using the Colab Files panel, or run the cell below to use the Files upload widget.

If you stored it on Google Drive, replace this with `!gdown <id>` instead.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick imu_actions_1000.jsonl
import os
assert os.path.exists('imu_actions_1000.jsonl'), 'upload imu_actions_1000.jsonl first'
print('rows in dataset:', sum(1 for _ in open('imu_actions_1000.jsonl', encoding='utf-8')))

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

## 3. Prompt template

Each JSONL row has a `messages` list with one user turn and one assistant turn. The user turn already carries the full instruction and the 14 IMU features for that 2 s window. The assistant turn is the action label.

We rewrap the messages into Nandakishor's `Input/Response` template so the trainer sees a single text field.

In [ ]:
input_prompt = """Below is an MPU6050 IMU window from a shoe mounted women safety chip. Classify the leg action.

### Input:
{}

### Response:
{}"""

## 4. Q-LoRA configuration

In [ ]:
# Base model: under 250M params, runs on Pi Zero W after Q4_K_M quantization
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Fine tuned model name
new_model = "smollm2-135m-safety-lora"

################################################################################
# QLoRA parameters
################################################################################
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05

################################################################################
# bitsandbytes parameters
################################################################################
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################
output_dir = "./results"
num_train_epochs = 5
fp16 = False
bf16 = False
per_device_train_batch_size = 8
per_device_eval_batch_size = 8
gradient_accumulation_steps = 1
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1
warmup_ratio = 0.03
group_by_length = True
save_steps = 0
logging_steps = 25

################################################################################
# SFT parameters
################################################################################
max_seq_length = 512
packing = False
device_map = {"": 0}

## 5. Build dataset and start fine tuning

In [ ]:
%cd /content
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
print(compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print('Your GPU supports bfloat16, you can set bf16=True for faster training')

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples['messages']:
        # messages list is [{'role':'user', ...}, {'role':'assistant', ...}]
        user_msg = next(m['content'] for m in msgs if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in msgs if m['role'] == 'assistant')
        text = input_prompt.format(user_msg, assistant_msg) + EOS_TOKEN
        texts.append(text)
    return {'text': texts}

dataset = load_dataset('json', data_files='imu_actions_1000.jsonl', split='train')
dataset = dataset.map(formatting_prompts_func, batched=True)
print('one formatted example:\n', dataset[0]['text'][:600])

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to='tensorboard',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

trainer.train()
trainer.model.save_pretrained(new_model)

## 6. Quick adapter inference test

In [ ]:
# Pick a synthetic IMU feature line that should be a hard kick
sample_user = (
    "You are an on chip safety monitor for a shoe mounted MPU6050. "
    "Given the IMU window features below classify the leg action into one of: "
    "standing, sitting, walking, running, jumping, stomp, shake_leg, soft_kick, hard_kick, repeated_kick. "
    "Reply with the label only.\n"
    "acc_mean_g=3.5 acc_std_g=5.2 acc_peak_g=16.0 acc_min_g=0.05 jerk_peak=560.0 jerk_mean=38.0 "
    "gyro_peak_dps=2000.0 gyro_std_dps=400.0 ax_std=2.5 ay_std=0.4 az_std=3.5 "
    "gx_std=10.0 gy_std=400.0 gz_std=8.0"
)
prompt = input_prompt.format(sample_user, '')
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=8, use_cache=True, do_sample=False)
text = tokenizer.batch_decode(outputs)[0]
print(text.split('### Response:')[1].split('###')[0].strip())

## 7. Merge LoRA adapter into the base model

In [ ]:
del model
del trainer
import gc; gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
merged = PeftModel.from_pretrained(base_model, new_model)
merged = merged.merge_and_unload()
merged_dir = 'final_weights_safety'
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print('saved to', merged_dir)

## 8. Convert to GGUF and quantize to Q4_K_M

In [ ]:
%cd /content
!git clone https://github.com/ggml-org/llama.cpp.git
%cd llama.cpp
!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j 8
!pip install -r requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
%cd /content/llama.cpp
!python convert_hf_to_gguf.py /content/final_weights_safety --outtype f16 --outfile /content/safety-f16.gguf

In [ ]:
%cd /content/llama.cpp
!./build/bin/llama-quantize /content/safety-f16.gguf /content/safety-q4_k_m.gguf q4_k_m
!ls -lh /content/safety-q4_k_m.gguf

## 9. End to end shoe firmware loop simulated with llama-cpp-python

This is the same code path that runs on the chip, only the binary is built for ARMv6 there. We classify a sequence of windows and let the trigger engine decide whether to fire the alarm.

In [ ]:
from llama_cpp import Llama
import json, time

llm = Llama(model_path='/content/safety-q4_k_m.gguf', n_gpu_layers=0, n_ctx=512, n_threads=4, verbose=False)

from collections import deque
KICKS = {'hard_kick', 'repeated_kick'}
history = deque(maxlen=3)

def classify(user_msg):
    p = input_prompt.format(user_msg, '')
    out = llm(p, max_tokens=8, temperature=0.0, top_p=1.0, stop=['###', '\n\n'])
    return out['choices'][0]['text'].strip().lower()

def trigger(label):
    history.append(label)
    if label == 'repeated_kick':
        return True
    return sum(1 for x in history if x in KICKS) >= 2

rows = [json.loads(l) for l in open('imu_actions_1000.jsonl', encoding='utf-8')]
demo = [r for r in rows if r['label'] in ('walking','hard_kick','hard_kick','repeated_kick')][:6]
for r in demo:
    user = next(m['content'] for m in r['messages'] if m['role']=='user')
    t0 = time.time()
    pred = classify(user)
    fire = trigger(pred)
    print(f"true={r['label']:14s} pred={pred:14s} fire={fire}  ({(time.time()-t0)*1000:.0f} ms)")

## 10. Save the gguf for the chip

Download `safety-q4_k_m.gguf` from the Files panel and copy it into the firmware build at `models/smollm2-135m-instruct-safety-q4_k_m.gguf`. The on chip wrapper in `slm_classifier.py` picks it up automatically through the `SAFETY_SLM_PATH` environment variable.

In [ ]:
from google.colab import files
files.download('/content/safety-q4_k_m.gguf')